✅ Trigger.AvailableNow

Waits briefly to check if new data arrives.
If data is available, it processes it in micro-batches.
If no data arrives within a short window, it automatically stops.
Best for: File-based sources (like S3, ADLS, HDFS) where files may appear at any time.

❌ Trigger.Once

Does not wait for new data.
Processes only the data that is already available at the time of execution.
If no data is present, it may result in no output and still terminate.
Best for: One-time ingestion from streaming sources like Kafka or Event Hub.

In [0]:
class Bronze_circuits():
    main_path="/Volumes/databricks_catalog/default/default_volume1"
    bronze_path = "streaming_project/bronze"

    def __init__(self,folder_name,source):
        self.folder_name=folder_name
        self.source=source
        
    def get_schema(self):
        schema=''' circuitId INT NOT NULL,
                circuitRef STRING NOT NULL,
                name STRING NOT NULL,
                location STRING,
                country STRING,
                lat DOUBLE,
                lng DOUBLE,
                alt INT,
                url STRING NOT NULL
                        '''
        return schema

    def read_data(self):
        df= (spark.readStream
            .format("cloudFiles") \
            .option("cloudFiles.format", "csv")
            .schema(self.get_schema())
            .option("maxFilesPerTrigger", 1)
            #.option("rowsPerSecond", 30)
            .option('header','true')
            .load(f"{self.main_path}/streaming_source/{self.folder_name}")
            )
        return df
        
    def process(self):
        print(f"\nStarting Bronze circuits Stream...", end='')
        readDF = self.read_data()
        from pyspark.sql.functions import current_timestamp,lit
        readDF= (readDF.withColumn('CircuitsIngestionDate',current_timestamp())
                 .withColumn('source',lit(self.source))
                 )
        sQuery =  ( readDF.writeStream
                            .queryName("bronze-ingestion-circuits")
                            .option("checkpointLocation", f"{self.main_path}/{self.bronze_path}/{self.folder_name}/checkpoint")
                            .outputMode("append")#full load so we need to use overwrite or complete mode n=but it is not supported by community edition
                            #.option('path',f"{self.main_path}/{self.bronze_path}/{self.folder_name}")
                            .trigger(availableNow=True)
                            .toTable('streaming_project.bronze.circuits')
                            #.start()     
                    ) 
        print("Done")
        return sQuery   


###lap_times

In [0]:
class Bronze_lap_times():
    main_path="/Volumes/databricks_catalog/default/default_volume1"
    bronze_path = "streaming_project/bronze" 
    
    def __init__(self,folder_name,source):
        self.folder_name=folder_name
        self.source=source
    
    def get_schema(self):
        schema='''
            raceId INT NOT NULL,
            driverId INT NOT NULL,
            lap INT NOT NULL,
            position INT,
            time STRING,
            milliseconds INT
        '''
        return schema

    def read_data(self):
        df= (spark.readStream
             .format("csv")
             .schema(self.get_schema())
             .option("maxFilesPerTrigger", 1)
             #.option("rowsPerSecond", 1000)
             .option('header','true')
             .load(f"{self.main_path}/streaming_source/{self.folder_name}")
             )
        return df
        
    def process(self):
        print(f"\nStarting Bronze lap_times Stream...", end='')
        readDF = self.read_data()
        from pyspark.sql.functions import current_timestamp,lit
        readDF= (readDF.withColumn('LapTimesIngestionDate',current_timestamp())
                 .withColumn('source',lit(self.source))
                 )
        sQuery =  ( readDF.writeStream
                            .queryName("bronze-ingestion-lap")
                            .option("checkpointLocation", f"{self.main_path}/{self.bronze_path}/{self.folder_name}/checkpoint")
                            .outputMode("append")
                            #.option('path',f"{self.main_path}/{self.bronze_path}/{self.folder_name}")
                            .trigger(availableNow=True)
                            .toTable('streaming_project.bronze.lap_times')
                    ) 
        print("Done")
        return sQuery   


###constructors

In [0]:
class Bronze_constructors():
    main_path="/Volumes/databricks_catalog/default/default_volume1"
    bronze_path = "streaming_project/bronze" 
    
    def __init__(self,folder_name,source):
        self.folder_name=folder_name
        self.source=source
    
    def get_schema(self):
        schema = '''constructorId INT NOT NULL,
            constructorRef STRING NOT NULL,
            name STRING NOT NULL,
            nationality STRING,
            url STRING NOT NULL
            '''

        return schema

    def read_data(self):
        df= (spark.readStream
             .format("json")
             .schema(self.get_schema())
             .option("maxFilesPerTrigger", 1)
             #.option("rowsPerSecond", 1000)
             .option('multiline','true')
             .load(f"{self.main_path}/streaming_source/{self.folder_name}")
             )
        return df
        
    def process(self):
        print(f"\nStarting Bronze constructor Stream...", end='')
        readDF = self.read_data()
        from pyspark.sql.functions import current_timestamp,lit
        readDF= (readDF.withColumn('ConstructorsIngestionDate',current_timestamp())
                 .withColumn('source',lit(self.source))
                 )
        sQuery =  ( readDF.writeStream
                            .queryName("bronze-ingestion-constructor")
                            .option("checkpointLocation", f"{self.main_path}/{self.bronze_path}/{self.folder_name}/checkpoint")
                            .outputMode("append")#full load so we need to use overwrite or complete mode n=but it is not supported by community edition
                            #.option('path',f"{self.main_path}/{self.bronze_path}/{self.folder_name}")
                            .trigger(availableNow=True)
                            .toTable('streaming_project.bronze.constructors')
                               
                    ) 
        print("Done")
        return sQuery   


###drivers

In [0]:
class Bronze_drivers():
    main_path="/Volumes/databricks_catalog/default/default_volume1"
    bronze_path = "streaming_project/bronze" 
    
    def __init__(self,folder_name,source):
        self.folder_name=folder_name
        self.source=source
    
    def get_schema(self):
        from pyspark.sql.types import IntegerType, StringType, DateType, StructType, StructField
        name_schema= StructType([
                StructField("forename", StringType(), False),        # not  Nullable
                StructField("surname", StringType(), False)           # not Nullable
                ])
    
        schema = StructType([
                StructField("driverId", IntegerType(), False),        # Primary Key, NOT NULL
                StructField("driverRef", StringType(), False),        # Unique identifier, NOT NULL
                StructField("number", IntegerType(), True),           # Nullable
                StructField("code", StringType(), True),              # Nullable
                StructField("name", name_schema, False),              # NOT NULL       
                StructField("dob", DateType(), True),                 # Nullable
                StructField("nationality", StringType(), True),       # Nullable
                StructField("url", StringType(), False)               # NOT NULL
                ])

        return schema

    def read_data(self):
        df= (spark.readStream
             .format("json")
             .schema(self.get_schema())
             .option("maxFilesPerTrigger", 1)
             #.option("rowsPerSecond", 1000)
             .load(f"{self.main_path}/streaming_source/{self.folder_name}")
             )
        return df
        
    def process(self):
        print(f"\nStarting Bronze dirvers Stream...", end='')
        readDF = self.read_data()
        from pyspark.sql.functions import current_timestamp,lit
        readDF= (readDF.withColumn('DriversIngestionDate',current_timestamp())
                 .withColumn('source',lit(self.source))
                 )
        sQuery =  ( readDF.writeStream
                            .queryName("bronze-ingestion-driver")
                            .option("checkpointLocation", f"{self.main_path}/{self.bronze_path}/{self.folder_name}/checkpoint")
                            .outputMode("append") #full load so we need to use overwrite or complete mode n=but it is not supported by community edition
                            #.option('path',f"{self.main_path}/{self.bronze_path}/{self.folder_name}")
                            .trigger(availableNow=True)
                            .toTable('streaming_project.bronze.drivers')
                    ) 
        print("Done")
        return sQuery   


###pit_stops

In [0]:
class Bronze_pit_stops():
    main_path="/Volumes/databricks_catalog/default/default_volume1"
    bronze_path = "streaming_project/bronze" 
    
    def __init__(self,folder_name,source):
        self.folder_name=folder_name
        self.source=source
    
    def get_schema(self):
        from pyspark.sql.types import IntegerType, StringType, DateType, StructType, StructField
        pit_stops_schema = StructType([
            StructField("raceId", IntegerType(), False),           # NOT NULL
            StructField("driverId", IntegerType(), False),         # NOT NULL
            StructField("stop", IntegerType(), False),             # NOT NULL
            StructField("lap", IntegerType(), False),              # NOT NULL
            StructField("time", StringType(), False),              # NOT NULL (time format like "13:52:25")
            StructField("duration", StringType(), True),           # Nullable
            StructField("milliseconds", IntegerType(), True)       # Nullable
        ])
        return pit_stops_schema

    def read_data(self):
        df= (spark.readStream
             .format("json")
             .schema(self.get_schema())
             .option('multiline','true')
             .option("maxFilesPerTrigger", 1)
             #.option("rowsPerSecond", 2000)
             .load(f"{self.main_path}/streaming_source/{self.folder_name}")
             )
        return df
        
    def process(self):
        from pyspark.sql.functions import current_timestamp,lit
        print(f"\nStarting Bronze pit_stops Stream...", end='')
        readDF = self.read_data()
        readDF= (readDF.withColumn('PitStopsIngestionDate',current_timestamp())
                 .withColumn('source',lit(self.source))
                 )
        sQuery =  ( readDF.writeStream
                            .queryName("bronze-ingestion-pit_stop")
                            .option("checkpointLocation", f"{self.main_path}/{self.bronze_path}/{self.folder_name}/checkpoint")
                            .outputMode("append")
                            #.option('path',f"{self.main_path}/{self.bronze_path}/{self.folder_name}")
                            .trigger(availableNow=True)
                            .toTable('streaming_project.bronze.pit_stops') 
                    ) 
        print("Done")
        return sQuery   


###qualifying

In [0]:
class Bronze_qualifying():
    main_path="/Volumes/databricks_catalog/default/default_volume1"
    bronze_path = "streaming_project/bronze" 
    
    def __init__(self,folder_name,source):
        self.folder_name=folder_name
        self.source=source
    
    def get_schema(self):
        from pyspark.sql.types import IntegerType, StringType, DateType, StructType, StructField
        qualifying_schema = StructType([
            StructField("qualifyId", IntegerType(), False),        # Primary Key, NOT NULL
            StructField("raceId", IntegerType(), False),           # Foreign Key, NOT NULL
            StructField("driverId", IntegerType(), False),         # Foreign Key, NOT NULL
            StructField("constructorId", IntegerType(), False),    # Foreign Key, NOT NULL
            StructField("number", IntegerType(), False),           # NOT NULL
            StructField("position", IntegerType(), True),          # Nullable
            StructField("q1", StringType(), True),                 # Nullable
            StructField("q2", StringType(), True),                 # Nullable
            StructField("q3", StringType(), True)                  # Nullable
        ])
        return qualifying_schema

    def read_data(self):
        df= (spark.readStream
             .format("json")
             .schema(self.get_schema())
             .option('multiline','true')
             .option("maxFilesPerTrigger", 1)
             #.option("rowsPerSecond", 1000)
             .load(f"{self.main_path}/streaming_source/{self.folder_name}")
             )
        return df
        
    def process(self):
        print(f"\nStarting Bronze qualifying Stream...", end='')
        readDF = self.read_data()
        from pyspark.sql.functions import current_timestamp,lit
        readDF= (readDF.withColumn('QualifyingIngestionDate',current_timestamp())
                 .withColumn('source',lit(self.source))
                 )
        sQuery =  ( readDF.writeStream
                            .queryName("bronze-ingestion-qualifying")
                            .option("checkpointLocation", f"{self.main_path}/{self.bronze_path}/{self.folder_name}/checkpoint")
                            .outputMode("append")
                            #.option('path',f"{self.main_path}/{self.bronze_path}/{self.folder_name}")
                            .trigger(availableNow=True)
                            .toTable('streaming_project.bronze.qualifying')
                                
                    ) 
        print("Done")
        return sQuery   


###races

In [0]:
class Bronze_races():
    main_path="/Volumes/databricks_catalog/default/default_volume1"
    bronze_path = "streaming_project/bronze" 
    
    def __init__(self,folder_name,source):
        self.folder_name=folder_name
        self.source=source
    
    def get_schema(self):
        from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType, TimeType
        races_schema = StructType([
            StructField("raceId", IntegerType(), nullable=False),
            StructField("year", IntegerType(), nullable=False),
            StructField("round", IntegerType(), nullable=False),
            StructField("circuitId", IntegerType(), nullable=False),
            StructField("name", StringType(), nullable=False),
            StructField("date", StringType(), nullable=False),
            StructField("time", StringType(), nullable=True),
            StructField("url", StringType(), nullable=True)
        ])
        return races_schema

    def read_data(self):
        df= (spark.readStream
             .format("csv")
             .option("header", True)
             .schema(self.get_schema())
             .option("maxFilesPerTrigger", 1)
             #.option("rowsPerSecond", 2000)
             .load(f"{self.main_path}/streaming_source/{self.folder_name}")
             )
        return df
        
    def process(self):
        print(f"\nStarting Bronze races Stream...", end='')
        readDF = self.read_data()
        from pyspark.sql.functions import current_timestamp,lit
        readDF= (readDF.withColumn('RacesIngestionDate',current_timestamp())
                 .withColumn('source',lit(self.source))
                 )
        sQuery =  ( readDF.writeStream
                            .queryName("bronze-ingestion-races")
                            .option("checkpointLocation", f"{self.main_path}/{self.bronze_path}/{self.folder_name}/checkpoint")
                            .outputMode("append")#full load so we need to use overwrite or complete mode n=but it is not supported by community edition
                            #.option('path',f"{self.main_path}/{self.bronze_path}/{self.folder_name}")
                            .trigger(availableNow=True)
                            .toTable('streaming_project.bronze.races')
                                 
                    ) 
        print("Done")
        return sQuery   


###results

In [0]:
class Bronze_results():
    main_path="/Volumes/databricks_catalog/default/default_volume1"
    bronze_path = "streaming_project/bronze" 
    
    def __init__(self,folder_name,source):
        self.folder_name=folder_name
        self.source=source
    
    def get_schema(self):
        results_schema = '''
                resultId INT NOT NULL,
                raceId INT NOT NULL,
                driverId INT NOT NULL,
                constructorId INT NOT NULL,
                number INT,
                grid INT NOT NULL,
                position INT,
                positionText STRING NOT NULL,
                positionOrder INT NOT NULL,
                points FLOAT NOT NULL,
                laps INT NOT NULL,
                time STRING,
                milliseconds INT,
                fastestLap INT,
                rank INT,
                fastestLapTime STRING,
                fastestLapSpeed STRING,
                statusId INT NOT NULL
            '''
        return results_schema

    def read_data(self):
        df= (spark.readStream
             .format("json")
             .schema(self.get_schema())
             .option('multiline','true')
             .option("maxFilesPerTrigger", 1)
             #.option("rowsPerSecond", 2000) #does nothing because auto loader does not read row by row streaming
             .load(f"{self.main_path}/streaming_source/{self.folder_name}")
             )
        return df
        
    def process(self):
        print(f"\nStarting Bronze results Stream...", end='')
        readDF = self.read_data()
        from pyspark.sql.functions import current_timestamp,lit
        readDF= (readDF.withColumn('ResultsIngestionDate',current_timestamp())
                 .withColumn('source',lit(self.source))
                 )
        sQuery =  ( readDF.writeStream
                            .queryName("bronze-ingestion-results")
                            .option("checkpointLocation", f"{self.main_path}/{self.bronze_path}/{self.folder_name}/checkpoint")
                            .outputMode("append")
                            #.option('path',f"{self.main_path}/{self.bronze_path}/{self.folder_name}")
                            .trigger(availableNow=True)
                            .toTable('streaming_project.bronze.results')
                                 
                    ) 
        print("Done")
        return sQuery   
